In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np

import pybedtools
from pybedtools import BedTool


#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data
import math
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))

#import importlib
#importlib.reload(const)

import const #to reload use import(importlib) and then importlib.reload(const)
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
const.set_plot_style()
import matplotlib.ticker as mtick

# --- Working directory (EDIT): used for relative .bed/.csv I/O in some notebooks ---
WORK_DIR = '/home/labs/davidgo/Collaboration/backup/humanMPRA/scripts/produce_paper_figures'
os.chdir(WORK_DIR)

output_path = '/home/labs/davidgo/Collaboration/humanMPRA/paper/raw_plots/fig_EVC/'

In [ ]:
L3a3_comb_df = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/L3a3/output/activity_after_filter/comb_df_adjusted_fdr.csv')


# produce EVC barcodes violinplots

In [ ]:
EVC_derived = L3a3_comb_df[L3a3_comb_df['oligo']=='seq_9777_chr4:5774930-5775199_Hh+SCREEN_derived_a3_L3']
EVC_ancestral = L3a3_comb_df[L3a3_comb_df['oligo']=='seq_9777_chr4:5774930-5775199_Hh+SCREEN_ancestral_a3_L3']
scrambled = L3a3_comb_df[L3a3_comb_df['oligo']=='scrambled_control_801_a3_L3']


In [ ]:
EVC_derived_rna, EVC_derived_dna, scrambled_rna = [], [], []
EVC_ancestral_rna, EVC_ancestral_dna,scrambled_dna = [], [], []

for c in ['RNA_chondrocytes_rep1','RNA_chondrocytes_rep2','RNA_chondrocytes_rep3']:
    EVC_derived_rna.extend(ast.literal_eval(EVC_derived.loc[EVC_derived.index[0], c]))

for c in ['DNA_chondrocytes_rep1','DNA_chondrocytes_rep2','DNA_chondrocytes_rep3']:
    EVC_derived_dna.extend(ast.literal_eval(EVC_derived.loc[EVC_derived.index[0], c]))

for c in ['RNA_chondrocytes_rep1','RNA_chondrocytes_rep2','RNA_chondrocytes_rep3']:
    EVC_ancestral_rna.extend(ast.literal_eval(EVC_ancestral.loc[EVC_ancestral.index[0], c]))

for c in ['DNA_chondrocytes_rep1','DNA_chondrocytes_rep2','DNA_chondrocytes_rep3']:
    EVC_ancestral_dna.extend(ast.literal_eval(EVC_ancestral.loc[EVC_ancestral.index[0], c]))

for c in ['RNA_chondrocytes_rep1','RNA_chondrocytes_rep2','RNA_chondrocytes_rep3']:
    scrambled_rna.extend(ast.literal_eval(scrambled.loc[scrambled.index[0], c]))

for c in ['DNA_chondrocytes_rep1','DNA_chondrocytes_rep2','DNA_chondrocytes_rep3']:
    scrambled_dna.extend(ast.literal_eval(scrambled.loc[scrambled.index[0], c]))

EVC_derived_ratio = [
    r / d
    for r, d in zip(EVC_derived_rna, EVC_derived_dna)
    if d != 0 and r!=0
]

EVC_ancestral_ratio = [
    r / d
    for r, d in zip(EVC_ancestral_rna, EVC_ancestral_dna)
    if d != 0 and r!=0
]

scrambled_ratio = [
    r / d
    for r, d in zip(scrambled_rna, scrambled_dna)
    if d != 0 and r!=0
]


In [ ]:
derived_df = pd.DataFrame({"value": EVC_derived_ratio}).assign(group="Barcodes of \nderived EVC allele")
ancestral_df = pd.DataFrame({"value": EVC_ancestral_ratio}).assign(group="Barcodes \nof ancestral EVC allele")
scrambled_df = pd.DataFrame({"value": scrambled_ratio}).assign(group="scrambled")
data_long = pd.concat([derived_df, ancestral_df,scrambled_df], ignore_index=True)


plt.figure(figsize=(6, 6))

# Jittered points
sns.stripplot(
    data=data_long, x="group", y="value",
    color="black", size=3, jitter = 0.35, alpha=0.1
)

palette = {
    "Barcodes of \nderived EVC allele": "#4798BD",
    "Barcodes \nof ancestral EVC allele": "#F5AF4C",
    "scrambled":"gray"
}

ax = sns.violinplot(
    data=data_long,
    x="group",
    y="value",
    inner="box",
    cut=0,
    palette=palette,
    linewidth = 0
)

ax = plt.gca()

yticks = ax.get_yticks()
plt.yticks([0, yticks[-1]])

plt.title("")
plt.ylabel("RNA DNA ratio (per barcode)")
plt.xlabel("")
plt.tight_layout()

const.save_fig(plt, 'EVC_violinplot', output_path)


plt.show()

In [ ]:
plt.figure(figsize=(6, 6))

sns.swarmplot(
    data=data_long,
    x="group",
    y="value",
    color="black",
    size=1,
    palette = palette,
    alpha=1
)


ax = plt.gca()

yticks = ax.get_yticks()
plt.yticks([0, yticks[-1]])

plt.title("")
plt.ylabel("RNA DNA ratio (per barcode)")
plt.xlabel("")
plt.tight_layout()

const.save_fig(plt, 'EVC_swarmplot', output_path)


plt.show()


In [ ]:
plt.figure(figsize=(8, 6))


sns.boxplot(
    data=data_long,
    x="group",
    y="value",
    palette = palette
)


ax = plt.gca()

yticks = ax.get_yticks()
plt.yticks([0, yticks[-1]])

plt.title("")
plt.ylabel("RNA DNA ratio (per barcode)")
plt.xlabel("")
plt.tight_layout()

const.save_fig(plt, 'EVC_boxplot', output_path)


plt.show()


# Genomic position enrichment analysis

In [ ]:
filter_minimum_oligo_counts_genomic_window =  20 # integer or None
filter_minimum_oligo_counts_TSS = 10
genomic_window_size = 5_000_000
TSS_window_size = 200_000

In [ ]:
if 'raw_data' not in locals():
    raw_data = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                        #usecols=range(0, 34), 
                        header=0)

    raw_data = raw_data.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
    raw_data = raw_data.loc[(raw_data['DNA_counts_raw_derived']>50) & (raw_data['DNA_counts_raw_ancestral']>50),:] #Filter based on counts

In [ ]:
chr_lengths = pd.read_csv(f'/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/hg19.chrom.sizes.txt', sep = '\t',
                     header=None)
chr_lengths.columns = ['chr','length']

# Define the canonical chromosomes you want
chrom_order = [f"chr{i}" for i in range(1, 23)] + ["chrX"]  # 1–22 + X

# Keep only these, drop everything else (contigs, chrY, chrM, etc.)
chr_lengths_filtered = (
    chr_lengths[chr_lengths["chr"].isin(chrom_order)]
    .copy()
)

# Enforce the order
chr_lengths_filtered = (
    chr_lengths_filtered
    .set_index("chr")
    .loc[chrom_order]
    .reset_index()
)
chrom_sizes = chr_lengths_filtered.rename(columns={"length": "length_bp"})


In [ ]:
oligos = raw_data.copy()
print(f'Oligo count before filtering = {len(oligos)}')

#oligos = oligos[abs(oligos['ATACseq_peaks_fetal_chondrocytes'])<1000]
#oligos = oligos[oligos['NCBI_Gene_ID'].notna()]
#oligos = oligos[oligos['within_promoter'].str.contains('T')]
print(f'Oligo count after filtering = {len(oligos)}')


In [ ]:
oligos_bed_df = oligos[["chromosome","start","end","oligo","differential_activity"]]
oligos_bed_df.loc[:,"start"] = oligos_bed_df["start"].map(int)
oligos_bed_df.loc[:,"end"] = oligos_bed_df["end"].map(int)
oligos_bed_df=oligos_bed_df.sort_values(['chromosome','start'], ascending = [True, True])

oligos_active_bed_df = oligos_bed_df[oligos_bed_df['differential_activity']==False][["chromosome","start","end"]]
oligos_diff_active_bed_df = oligos_bed_df[oligos_bed_df['differential_activity']==True][["chromosome","start","end"]]
oligos_bed = pybedtools.BedTool.from_dataframe(oligos_bed_df)
oligos_active_bed = pybedtools.BedTool.from_dataframe(oligos_active_bed_df)
oligos_diff_active_bed = pybedtools.BedTool.from_dataframe(oligos_diff_active_bed_df)


In [ ]:
# -------------------------------------------------------
# STARTING POINT: You already have these
# -------------------------------------------------------
# active   = BedTool("active.bed")
# diff     = BedTool("diff_active.bed")

# Convert to pandas
active_df = oligos_active_bed.to_dataframe(names=["chr", "start", "end"])
diff_df   = oligos_diff_active_bed.to_dataframe(names=["chr", "start", "end"])

# Add midpoint for plotting
active_df["mid"] = (active_df["start"] + active_df["end"]) / 2
diff_df["mid"]   = (diff_df["start"] + diff_df["end"]) / 2

In [ ]:
# ------------------------------------------
# Build cumulative chromosome offsets
# ------------------------------------------
chrom_offsets = {}
offset = 0
for _, row in chrom_sizes.iterrows():
    chrom = row["chr"]
    chrom_offsets[chrom] = offset
    offset += row["length_bp"]


# ------------------------------------------
# Assign bins per chromosome
# ------------------------------------------
def assign_bins(df, genomic_window_size=genomic_window_size):
    """
    Add a 'bin' column based on the midpoint position.
    Assumes df has columns: 'chr', 'mid'
    """
    df = df.copy()
    df["bin"] = (df["mid"] // genomic_window_size).astype(int)
    return df

# diff_df: differential-active oligos
# active_df: active but NOT differential
diff_binned   = assign_bins(diff_df)
active_binned = assign_bins(active_df)

# ------------------------------------------
# Per-bin counts: a = diff-active, b = non-diff active
# ------------------------------------------
a_counts = (
    diff_binned
    .groupby(["chr", "bin"])
    .size()
    .reset_index(name="a")     # a = diff-active in bin
)

b_counts = (
    active_binned
    .groupby(["chr", "bin"])
    .size()
    .reset_index(name="b")     # b = active non-diff in bin
)

# Merge to get both counts per bin (keep bins that appear in either)
bins = (
    pd.merge(a_counts, b_counts, on=["chr", "bin"], how="outer")
    .fillna(0)
)

# Make sure counts are integers
bins["a"] = bins["a"].astype(int)
bins["b"] = bins["b"].astype(int)

# Total active in bin (for convenience / plotting)
bins["active_total"] = bins["a"] + bins["b"]

# Genomic coordinates of each bin (within chromosome)
bins["start"] = bins["bin"] * genomic_window_size
bins["end"]   = (bins["bin"] + 1) * genomic_window_size


In [ ]:
# ------------------------------------------
# Enrichment: Fisher's exact test per bin
# ------------------------------------------
# diff-active oligos are a subset of active oligos.
#
#             diff-active     not-diff
# in bin         a              b
# outside        c              d

# Global totals from the per-bin counts
total_diff     = bins["a"].sum()   # all diff-active
total_non_diff = bins["b"].sum()   # all active but non-diff

pvals = []
odds_ratios = []

for idx, row in bins.iterrows():
    a = row["a"]  # diff-active in bin
    b = row["b"]  # non-diff active in bin

    # Outside-bin counts
    c = total_diff     - a   # diff-active outside bin
    d = total_non_diff - b   # non-diff active outside bin

    table = [[a, b],
             [c, d]]

    orat, pval = fisher_exact(table,alternative='greater')
    odds_ratios.append(orat)
    pvals.append(pval)

bins["odds_ratio"] = odds_ratios
bins["pval"]       = pvals


In [ ]:
# ------------------------------------------
# Multiple testing correction (BH / FDR)
# ------------------------------------------

# Mask of bins to include in FDR correction
mask = bins["active_total"] >= filter_minimum_oligo_counts_genomic_window
print(f"Passed filter: {sum(mask)}")
# Run BH only on selected p-values
reject_sub, p_adj_sub, _, _ = multipletests(
    bins.loc[mask, "pval"].values,
    alpha=0.05,
    method="fdr_bh"
)

# Initialize with defaults
bins["p_adj"] = np.nan
bins["significant"] = False

# Fill in only for tested bins
bins.loc[mask, "p_adj"] = p_adj_sub
bins.loc[mask, "significant"] = reject_sub


bins_sorted = bins.sort_values(["chr", "bin"]).copy()

# Bin midpoint within chromosome
bins_sorted["bin_mid"] = bins_sorted["start"] + genomic_window_size / 2

# Add chromosome offset and global coordinate
bins_sorted["chr_offset"] = bins_sorted["chr"].map(chrom_offsets)
bins_sorted["genome_pos"] = bins_sorted["chr_offset"] + bins_sorted["bin_mid"]

# Precompute -log10(FDR) for convenience
bins_sorted["minus_log10_fdr"] = -np.log10(
    bins_sorted["p_adj"].replace(0, np.nextafter(0, 1))
)

In [ ]:
# ===========================================================
# Volcano plot: log2(enrichment_ratio) vs -log10(FDR)
# ===========================================================

bins_plot = bins_sorted.copy()

# x-axis: log2 enrichment
bins_plot["log2_enrichment"] = np.log2(bins_plot["odds_ratio"].replace([0, np.inf], np.nan))

# y-axis: -log10(FDR)
bins_plot["minus_log10_fdr"] = -np.log10(bins_plot["p_adj"].replace(0, np.nextafter(0, 1)))

# threshold for significance
sig_mask = bins_plot["p_adj"] < 0.05

# enriched (ratio > 1)
enriched_mask = sig_mask & (bins_plot["log2_enrichment"] > 0)

# depleted (ratio < 1)
depleted_mask = sig_mask & (bins_plot["log2_enrichment"] < 0)

plt.figure(figsize=(10, 7))

# non-significant points
plt.scatter(
    bins_plot.loc[~sig_mask, "log2_enrichment"],
    bins_plot.loc[~sig_mask, "minus_log10_fdr"],
    color="gray",
    alpha=0.5,
    s=12,
    label="Not significant"
)

# significantly enriched
plt.scatter(
    bins_plot.loc[enriched_mask, "log2_enrichment"],
    bins_plot.loc[enriched_mask, "minus_log10_fdr"],
    color="tomato",
    alpha=0.8,
    s=20,
    label="Enriched (FDR < 0.05)"
)

# significantly depleted
plt.scatter(
    bins_plot.loc[depleted_mask, "log2_enrichment"],
    bins_plot.loc[depleted_mask, "minus_log10_fdr"],
    color="steelblue",
    alpha=0.8,
    s=20,
    label="Depleted (FDR < 0.05)"
)

plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=1)

plt.xlabel("log2( enrichment_ratio )")
plt.ylabel("-log10( FDR )")
plt.title("Volcano Plot — 1 Mb regions enriched for differential-active oligos")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# Manhattan plot of -log10(FDR) across genome
# ==========================================

# -log10(FDR), protect against 0
bins_sorted["minus_log10_fdr"] = -np.log10(
    bins_sorted["p_adj"].replace(0, np.nextafter(0, 1))
)
bins_sorted["chrom_clean"] = bins_sorted["chr"].str.replace("^chr", "", regex=True)
chrom_order = [f"{i}" for i in range(1, 23)] + ["X"]  # 1–22 + X

plt.figure(figsize=(14, 5))

# Alternate colors per chromosome
chrom_colors = ["#666666", "#B3B3B3"]  # or any two colors you like

xticks = []
xtick_labels = []

for i, chrom in enumerate(chrom_order):
    sub = bins_sorted[bins_sorted["chrom_clean"] == chrom]
    if sub.empty:
        continue

    # scatter points for this chromosome
    plt.bar(
        sub["genome_pos"],
        sub["minus_log10_fdr"],
        width=5000000,                 # adjust! (see note below)
        alpha=0.8,
        color=chrom_colors[i % 2],
        align="center",
        linewidth=0,
    )


        # scatter points for this chromosome
    plt.scatter(
        sub["genome_pos"],
        sub["minus_log10_fdr"],
        s=50,                 # adjust! (see note below)
        alpha=1,
        color=chrom_colors[i % 2],
        linewidth=0,
    )

    # chromosome label at its midpoint
    mid_pos = (sub["genome_pos"].min() + sub["genome_pos"].max()) / 2
    xticks.append(mid_pos)
    xtick_labels.append(chrom)
    
    # Optional: highlight significant bins in a different color
    sig_mask = bins_sorted["p_adj"] < 0.05
    plt.scatter(
        bins_sorted.loc[sig_mask, "genome_pos"],
        bins_sorted.loc[sig_mask, "minus_log10_fdr"],
        s=50,
        color="tomato",
        alpha=1,
        #label="FDR < 0.05",
    )
    #plt.legend()


# significance line at FDR = 0.05
sig_line = -np.log10(0.05)
plt.axhline(sig_line, linestyle="--", linewidth=1, color="black")

yticks = plt.yticks()[0]
plt.yticks([yticks[0], yticks[-2]])

plt.xticks(xticks, xtick_labels, rotation=90)
plt.ylabel("-log10(FDR)")
plt.xlabel("Chromosome")
#plt.title("Manhattan plot — 1 Mb bins enriched for differential-active oligos")
plt.tight_layout()
plt.margins(x=0.005)
plt.ylim(-0.05,2.8)
const.save_fig(plt, 'genomic_window_enrichment_analysis', output_path)

plt.show()


In [ ]:
# ===================================================================
# Load transcript annotations and create gene regions
# ===================================================================

# Load NCBI transcripts BED file
transcripts_df = pd.read_csv(
    '/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/NCBI/hg19/gff_parsed/NCBI_transcripts_hg19.bed',
    sep='\t',
    header=None,
    names=['chr', 'start', 'end', 'strand', 'gene', 'gene_id', 'transcript_id', 'quality', 'feature_type', 'annotations']
)

# Filter to only coding transcripts (mRNA = protein-coding)
coding_transcripts = transcripts_df[transcripts_df['feature_type'] == 'mRNA'].copy()

# Collapse transcripts to single gene regions (min start, max end per gene per chromosome)
gene_regions = coding_transcripts.groupby(['chr', 'gene']).agg({
    'start': 'min',
    'end': 'max',
    'strand': 'first'
}).reset_index()

print(f"Loaded {len(transcripts_df)} transcripts")
print(f"Coding transcripts (mRNA): {len(coding_transcripts)}")
print(f"Unique genes: {len(gene_regions)}")
print(f"Chr4 coding genes: {len(gene_regions[gene_regions['chr'] == 'chr4'])}")

# ===================================================================
# Function to annotate genes on density plot
# ===================================================================

def annotate_genes_on_plot(ax, chromosome, start_pos, end_pos, gene_regions_df, y_position, height=0.02, label_rotation=90, max_labels=15):
    """
    Add gene annotations to a matplotlib axis.
    
    Parameters:
    - ax: matplotlib axis
    - chromosome: chromosome to filter genes (e.g., 'chr4')
    - start_pos: leftmost position in plot
    - end_pos: rightmost position in plot
    - gene_regions_df: dataframe with columns 'chr', 'gene', 'start', 'end'
    - y_position: y-data position to place gene boxes
    - height: height of gene boxes
    - label_rotation: rotation angle for gene labels
    - max_labels: maximum number of gene labels to display
    """
    
    from matplotlib.patches import Rectangle
    
    # Filter genes in region
    genes_in_region = gene_regions_df[
        (gene_regions_df['chr'] == chromosome) &
        (gene_regions_df['start'] < end_pos) &
        (gene_regions_df['end'] > start_pos)
    ].copy()
    
    if len(genes_in_region) == 0:
        return
    
    # Sort by size (larger genes first) and take top max_labels
    genes_in_region['size'] = genes_in_region['end'] - genes_in_region['start']
    genes_to_label = genes_in_region.nlargest(max_labels, 'size')
    
    # Draw rectangles for all genes
    for _, gene in genes_in_region.iterrows():
        rect = Rectangle(
            (gene['start'], y_position),
            gene['end'] - gene['start'],
            height,
            alpha=0.1,
            color='navy',
            linewidth=0
        )
        ax.add_patch(rect)
    
    # Add labels for largest genes only
    for _, gene in genes_to_label.iterrows():
        mid_pos = (gene['start'] + gene['end']) / 2
        ax.text(mid_pos, y_position + height/2, gene['gene'], 
                rotation=label_rotation, ha='center', va='center',
                fontsize=7, alpha=0.8, fontweight='bold')

# ===================================================================
# EVC region density plot - non-active, active, and diff-active
# ===================================================================

# Define EVC gene region and 5 Mb window
evc_start = 5_712_928
evc_end = 5_816_032


evc_center = (evc_start + evc_end) / 2
window_half = 5_000_000  # 2.5 Mb on each side = 5 Mb total window
window_start = int(evc_center - window_half)
window_end = int(evc_center + window_half)

# Filter to chr4 and the window region
diff_evc = diff_df[(diff_df['chr'] == 'chr4') &
                   (diff_df['start'] >= window_start) &
                   (diff_df['start'] <= window_end)].copy()

active_evc = active_df[(active_df['chr'] == 'chr4') &
                       (active_df['start'] >= window_start) &
                       (active_df['start'] <= window_end)].copy()

# Prepare oligos_bed_df by renaming 'chromosome' to 'chr' for consistency
oligos_for_concat = oligos_bed_df[['chromosome', 'start', 'end']].copy()
oligos_for_concat.rename(columns={'chromosome': 'chr'}, inplace=True)

all_evc = pd.concat([diff_evc, active_evc, oligos_for_concat[
    (oligos_for_concat['chr'] == 'chr4') &
    (oligos_for_concat['start'] >= window_start) &
    (oligos_for_concat['start'] <= window_end)
]]).drop_duplicates(['chr', 'start', 'end'])

# Build set of active/diff-active for filtering
active_union_set = set(zip(active_evc['chr'], active_evc['start'], active_evc['end']))
active_union_set.update(zip(diff_evc['chr'], diff_evc['start'], diff_evc['end']))

# Compute non-active as everything minus active/diff-active
non_active_evc = all_evc[~all_evc.apply(lambda row: (row['chr'], row['start'], row['end']) in active_union_set, axis=1)]

# Extract positions
non_active_positions = non_active_evc['start'].values
active_positions = active_evc['start'].values
diff_positions = diff_evc['start'].values

print(f"EVC region variants: non-active={len(non_active_positions)}, active={len(active_positions)}, diff-active={len(diff_positions)}")

# Create densities using KDE
x_range = np.linspace(window_start, window_end, 10000)
kde_non_active = gaussian_kde(non_active_positions, bw_method=0.05)
kde_active = gaussian_kde(active_positions, bw_method=0.05)
kde_diff = gaussian_kde(diff_positions, bw_method=0.05)

density_non_active = kde_non_active(x_range)
density_active = kde_active(x_range)
density_diff = kde_diff(x_range)

# Create plot
fig, ax = plt.subplots(figsize=(14, 7))

# Plot densities
ax.fill_between(x_range, density_non_active, alpha=0.3, color='gray', label='Non-active')
ax.plot(x_range, density_non_active, color='gray', linewidth=1.5, alpha=0.8)

ax.fill_between(x_range, density_active, alpha=0.3, color='steelblue', label='Active')
ax.plot(x_range, density_active, color='steelblue', linewidth=1.5, alpha=0.8)

ax.fill_between(x_range, density_diff, alpha=0.3, color='tomato', label='Differential-active')
ax.plot(x_range, density_diff, color='tomato', linewidth=1.5, alpha=0.8)

# Highlight EVC gene region
ax.axvspan(evc_start, evc_end, alpha=0.15, color='green', label='EVC gene')

# Add rug plot for individual variants at bottom
y_lim = ax.get_ylim()
y_rug = y_lim[0] - (y_lim[1] - y_lim[0]) * 0.05

ax.scatter(non_active_positions, [y_rug]*len(non_active_positions), color='gray', s=20, alpha=0.4, marker='_', linewidths=0)
ax.scatter(active_positions, [y_rug]*len(active_positions), color='steelblue', s=30, alpha=0.6, marker='o', linewidths=0)
ax.scatter(diff_positions, [y_rug]*len(diff_positions), color='tomato', s=30, alpha=0.6, marker='o', linewidths=0)

# Update y-axis to accommodate the rug plot
ax.set_ylim(y_lim[0] - (y_lim[1] - y_lim[0]) * 0.12, y_lim[1])

# Add gene annotations below the plot area
# Compute space for gene annotations
gene_y_min = y_lim[0] - (y_lim[1] - y_lim[0]) * 0.1
gene_height = (y_lim[1] - y_lim[0]) * 0.08

annotate_genes_on_plot(
    ax,
    chromosome='chr4',
    start_pos=window_start,
    end_pos=window_end,
    gene_regions_df=gene_regions,
    y_position=gene_y_min,
    height=gene_height,
    max_labels=15
)

# Adjust y-axis to show gene boxes
ax.set_ylim(gene_y_min - gene_height, y_lim[1])

# Format axes
ax.set_xlabel('Genomic Position (chr4)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Variant Density Surrounding EVC Gene (5 Mb window)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)

# Format x-axis with comma-separated thousands
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ===================================================================
# Load transcript annotations and create gene regions
# ===================================================================

# Load NCBI transcripts BED file
transcripts_df = pd.read_csv(
    '/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/NCBI/hg19/gff_parsed/NCBI_transcripts_hg19.bed',
    sep='\t',
    header=None,
    names=['chr', 'start', 'end', 'strand', 'gene', 'gene_id', 'transcript_id', 'quality', 'feature_type', 'annotations']
)

# Filter to only coding transcripts (mRNA = protein-coding)
coding_transcripts = transcripts_df[transcripts_df['feature_type'] == 'mRNA'].copy()

# Collapse transcripts to single gene regions (min start, max end per gene per chromosome)
gene_regions = coding_transcripts.groupby(['chr', 'gene']).agg({
    'start': 'min',
    'end': 'max',
    'strand': 'first'
}).reset_index()

print(f"Loaded {len(transcripts_df)} transcripts")
print(f"Coding transcripts (mRNA): {len(coding_transcripts)}")
print(f"Unique genes: {len(gene_regions)}")
print(f"Chr4 coding genes: {len(gene_regions[gene_regions['chr'] == 'chr4'])}")

# ===================================================================
# Function to annotate genes on density plot
# ===================================================================

def annotate_genes_on_plot(ax, chromosome, start_pos, end_pos, gene_regions_df, y_position, height=0.02, label_rotation=90, max_labels=15):
    """
    Add gene annotations to a matplotlib axis.
    
    Parameters:
    - ax: matplotlib axis
    - chromosome: chromosome to filter genes (e.g., 'chr4')
    - start_pos: leftmost position in plot
    - end_pos: rightmost position in plot
    - gene_regions_df: dataframe with columns 'chr', 'gene', 'start', 'end'
    - y_position: y-data position to place gene boxes
    - height: height of gene boxes
    - label_rotation: rotation angle for gene labels
    - max_labels: maximum number of gene labels to display
    """
    
    from matplotlib.patches import Rectangle
    
    # Filter genes in region
    genes_in_region = gene_regions_df[
        (gene_regions_df['chr'] == chromosome) &
        (gene_regions_df['start'] < end_pos) &
        (gene_regions_df['end'] > start_pos)
    ].copy()
    
    if len(genes_in_region) == 0:
        return
    
    # Sort by size (larger genes first) and take top max_labels
    genes_in_region['size'] = genes_in_region['end'] - genes_in_region['start']
    genes_to_label = genes_in_region.nlargest(max_labels, 'size')
    
    # Draw rectangles for all genes
    for _, gene in genes_in_region.iterrows():
        rect = Rectangle(
            (gene['start'], y_position),
            gene['end'] - gene['start'],
            height,
            alpha=0.1,
            color='navy',
            linewidth=0
        )
        ax.add_patch(rect)
    
    # Add labels for largest genes only
    for _, gene in genes_to_label.iterrows():
        mid_pos = (gene['start'] + gene['end']) / 2
        ax.text(mid_pos, y_position + height/2, gene['gene'], 
                rotation=label_rotation, ha='center', va='center',
                fontsize=7, alpha=0.8, fontweight='bold')

# ===================================================================
# EVC region - sliding window variant count
# ===================================================================

# Define EVC gene region and 5 Mb window
evc_start = 5_712_928
evc_end = 5_816_032

evc_center = (evc_start + evc_end) / 2
window_half = 5_000_000  # 2.5 Mb on each side = 5 Mb total window
window_start = int(evc_center - window_half)
window_end = int(evc_center + window_half)

# Filter to chr4 and the window region
diff_evc = diff_df[(diff_df['chr'] == 'chr4') &
                   (diff_df['start'] >= window_start) &
                   (diff_df['start'] <= window_end)].copy()

active_evc = active_df[(active_df['chr'] == 'chr4') &
                       (active_df['start'] >= window_start) &
                       (active_df['start'] <= window_end)].copy()

# Prepare oligos_bed_df by renaming 'chromosome' to 'chr' for consistency
oligos_for_concat = oligos_bed_df[['chromosome', 'start', 'end']].copy()
oligos_for_concat.rename(columns={'chromosome': 'chr'}, inplace=True)

all_evc = pd.concat([diff_evc, active_evc, oligos_for_concat[
    (oligos_for_concat['chr'] == 'chr4') &
    (oligos_for_concat['start'] >= window_start) &
    (oligos_for_concat['start'] <= window_end)
]]).drop_duplicates(['chr', 'start', 'end'])

# Build set of active/diff-active for filtering
active_union_set = set(zip(active_evc['chr'], active_evc['start'], active_evc['end']))
active_union_set.update(zip(diff_evc['chr'], diff_evc['start'], diff_evc['end']))

# Compute non-active as everything minus active/diff-active
non_active_evc = all_evc[~all_evc.apply(lambda row: (row['chr'], row['start'], row['end']) in active_union_set, axis=1)]

# Extract positions
non_active_positions = non_active_evc['start'].values
active_positions = active_evc['start'].values
diff_positions = diff_evc['start'].values

print(f"EVC region variants: non-active={len(non_active_positions)}, active={len(active_positions)}, diff-active={len(diff_positions)}")

# ===================================================================
# Sliding window approach - count variants in each window
# ===================================================================

# Define sliding window parameters (ADJUST THESE AS NEEDED)
window_size = 500_000  # window size in bp
window_step = 250_000  # step size (250k = 50% overlap with 500k window)

# Create window midpoints
window_positions = np.arange(window_start + window_size/2, window_end, window_step)

# Count variants in each window for the three categories
non_active_counts = []
active_counts = []
diff_counts = []

for mid in window_positions:
    w_start = mid - window_size/2
    w_end = mid + window_size/2
    
    non_active_counts.append(np.sum((non_active_positions >= w_start) & (non_active_positions <= w_end)))
    active_counts.append(np.sum((active_positions >= w_start) & (active_positions <= w_end)))
    diff_counts.append(np.sum((diff_positions >= w_start) & (diff_positions <= w_end)))

non_active_counts = np.array(non_active_counts)
active_counts = np.array(active_counts)
diff_counts = np.array(diff_counts)

# Create plot
fig, ax = plt.subplots(figsize=(14, 7))

# Plot stacked bars
bar_width = window_step * 0.85
ax.bar(window_positions, non_active_counts, width=bar_width, alpha=0.7, color='gray', label='Non-active')
ax.bar(window_positions, active_counts, width=bar_width, alpha=0.7, color='steelblue', label='Active', 
       bottom=non_active_counts)
ax.bar(window_positions, diff_counts, width=bar_width, alpha=0.7, color='tomato', label='Differential-active',
       bottom=non_active_counts + active_counts)

# Highlight EVC gene region
ax.axvspan(evc_start, evc_end, alpha=0.15, color='green', label='EVC gene')

# Update y-axis for gene annotations
y_lim = ax.get_ylim()

# Add gene annotations below the plot area
gene_y_min = y_lim[0] - (y_lim[1] - y_lim[0]) * 0.15
gene_height = (y_lim[1] - y_lim[0]) * 0.08

annotate_genes_on_plot(
    ax,
    chromosome='chr4',
    start_pos=window_start,
    end_pos=window_end,
    gene_regions_df=gene_regions,
    y_position=gene_y_min,
    height=gene_height,
    max_labels=15
)

# Adjust y-axis to show gene boxes
ax.set_ylim(gene_y_min - gene_height, y_lim[1])

# Format axes
ax.set_xlabel(f'Genomic Position (chr4) - Window: {window_size/1e6:.1f} Mb, Step: {window_step/1e6:.1f} Mb', fontsize=12)
ax.set_ylabel('Variant Count', fontsize=12)
ax.set_title('Variant Count in Sliding Windows - EVC Region', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)

# Format x-axis with comma-separated thousands
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ===================================================================
# CUSTOM REGION DENSITY PLOT - configurable region of interest
# ===================================================================

# *** USER: Define your region of interest here ***
region_chr = 'chr1'
region_start = 85_000_000  # Edit this
region_end = 125_000_000    # Edit this
region_window_half = 5_000_000  # Size of window around region (total window = 2 * this)

# Calculate window boundaries
region_center = (region_start + region_end) / 2
window_start = int(region_center - region_window_half)
window_end = int(region_center + region_window_half)

# Filter to specified chromosome and window region
diff_custom = diff_df[(diff_df['chr'] == region_chr) &
                      (diff_df['start'] >= window_start) &
                      (diff_df['start'] <= window_end)].copy()

active_custom = active_df[(active_df['chr'] == region_chr) &
                          (active_df['start'] >= window_start) &
                          (active_df['start'] <= window_end)].copy()

# Prepare oligos_bed_df by renaming 'chromosome' to 'chr' for consistency
oligos_for_concat_custom = oligos_bed_df[['chromosome', 'start', 'end']].copy()
oligos_for_concat_custom.rename(columns={'chromosome': 'chr'}, inplace=True)

all_custom = pd.concat([diff_custom, active_custom, oligos_for_concat_custom[
    (oligos_for_concat_custom['chr'] == region_chr) &
    (oligos_for_concat_custom['start'] >= window_start) &
    (oligos_for_concat_custom['start'] <= window_end)
]]).drop_duplicates(['chr', 'start', 'end'])

# Build set of active/diff-active for filtering
active_union_set_custom = set(zip(active_custom['chr'], active_custom['start'], active_custom['end']))
active_union_set_custom.update(zip(diff_custom['chr'], diff_custom['start'], diff_custom['end']))

# Compute non-active as everything minus active/diff-active
non_active_custom = all_custom[~all_custom.apply(lambda row: (row['chr'], row['start'], row['end']) in active_union_set_custom, axis=1)]

# Extract positions
non_active_positions_custom = non_active_custom['start'].values
active_positions_custom = active_custom['start'].values
diff_positions_custom = diff_custom['start'].values

print(f"Custom region ({region_chr}:{region_start:,}-{region_end:,}) variants:")
print(f"  non-active={len(non_active_positions_custom)}, active={len(active_positions_custom)}, diff-active={len(diff_positions_custom)}")

# Create densities using KDE
x_range_custom = np.linspace(window_start, window_end, 10000)
kde_non_active_custom = gaussian_kde(non_active_positions_custom, bw_method=0.05)
kde_active_custom = gaussian_kde(active_positions_custom, bw_method=0.05)
kde_diff_custom = gaussian_kde(diff_positions_custom, bw_method=0.05)

density_non_active_custom = kde_non_active_custom(x_range_custom)
density_active_custom = kde_active_custom(x_range_custom)
density_diff_custom = kde_diff_custom(x_range_custom)

# Create plot
fig, ax = plt.subplots(figsize=(14, 7))

# Plot densities
ax.fill_between(x_range_custom, density_non_active_custom, alpha=0.3, color='gray', label='Non-active')
ax.plot(x_range_custom, density_non_active_custom, color='gray', linewidth=1.5, alpha=0.8)

ax.fill_between(x_range_custom, density_active_custom, alpha=0.3, color='steelblue', label='Active')
ax.plot(x_range_custom, density_active_custom, color='steelblue', linewidth=1.5, alpha=0.8)

ax.fill_between(x_range_custom, density_diff_custom, alpha=0.3, color='tomato', label='Differential-active')
ax.plot(x_range_custom, density_diff_custom, color='tomato', linewidth=1.5, alpha=0.8)

# Add rug plot for individual variants at bottom
y_lim = ax.get_ylim()
y_rug = y_lim[0] - (y_lim[1] - y_lim[0]) * 0.05

ax.scatter(non_active_positions_custom, [y_rug]*len(non_active_positions_custom), color='gray', s=20, alpha=0.4, marker='_', linewidths=0)
ax.scatter(active_positions_custom, [y_rug]*len(active_positions_custom), color='steelblue', s=30, alpha=0.6, marker='o', linewidths=0)
ax.scatter(diff_positions_custom, [y_rug]*len(diff_positions_custom), color='tomato', s=30, alpha=0.6, marker='o', linewidths=0)

# Update y-axis to accommodate the rug plot
ax.set_ylim(y_lim[0] - (y_lim[1] - y_lim[0]) * 0.12, y_lim[1])

# Add gene annotations below the plot area
gene_y_min = y_lim[0] - (y_lim[1] - y_lim[0]) * 0.1
gene_height = (y_lim[1] - y_lim[0]) * 0.08
annotate_genes_on_plot(
    ax,
    chromosome=region_chr,
    start_pos=window_start,
    end_pos=window_end,
    gene_regions_df=gene_regions,
    y_position=gene_y_min,
    height=gene_height,
    max_labels=15
)

# Adjust y-axis to show gene boxes
ax.set_ylim(gene_y_min - gene_height, y_lim[1])

# Format axes
ax.set_xlabel(f'Genomic Position ({region_chr})', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Variant Density Surrounding Region {region_chr}:{region_start:,}-{region_end:,}', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)

# Format x-axis with comma-separated thousands
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ===================================================================
# Chromosome 4-wide density plot - non-active, active, and diff-active
# ===================================================================

# Filter to chromosome 4
diff_chr4 = diff_df[diff_df['chr'] == 'chr4'].copy()
active_chr4 = active_df[active_df['chr'] == 'chr4'].copy()

# Prepare oligos_bed_df by renaming 'chromosome' to 'chr' for consistency
oligos_for_concat = oligos_bed_df[['chromosome', 'start', 'end']].copy()
oligos_for_concat.rename(columns={'chromosome': 'chr'}, inplace=True)

all_chr4 = pd.concat([diff_chr4, active_chr4, oligos_for_concat[oligos_for_concat['chr'] == 'chr4']]).drop_duplicates(['chr', 'start', 'end'])

# Build set of active/diff-active for filtering
active_union_set_chr4 = set(zip(active_chr4['chr'], active_chr4['start'], active_chr4['end']))
active_union_set_chr4.update(zip(diff_chr4['chr'], diff_chr4['start'], diff_chr4['end']))

# Compute non-active as everything minus active/diff-active
non_active_chr4 = all_chr4[~all_chr4.apply(lambda row: (row['chr'], row['start'], row['end']) in active_union_set_chr4, axis=1)]

# Extract positions
non_active_positions_chr4 = non_active_chr4['start'].values
active_positions_chr4 = active_chr4['start'].values
diff_positions_chr4 = diff_chr4['start'].values

print(f"Chr4 variants: non-active={len(non_active_positions_chr4)}, active={len(active_positions_chr4)}, diff-active={len(diff_positions_chr4)}")

# Get chromosome 4 length
chr4_length = chrom_sizes[chrom_sizes['chr'] == 'chr4']['length_bp'].values[0]

# Create densities using KDE with smaller bandwidth for chromosome-scale
x_range_chr4 = np.linspace(0, chr4_length, 20000)
kde_non_active_chr4 = gaussian_kde(non_active_positions_chr4, bw_method=0.001)
kde_active_chr4 = gaussian_kde(active_positions_chr4, bw_method=0.001)
kde_diff_chr4 = gaussian_kde(diff_positions_chr4, bw_method=0.001)

density_non_active_chr4 = kde_non_active_chr4(x_range_chr4)
density_active_chr4 = kde_active_chr4(x_range_chr4)
density_diff_chr4 = kde_diff_chr4(x_range_chr4)

# Create plot
fig, ax = plt.subplots(figsize=(14, 7))

# Plot densities
ax.fill_between(x_range_chr4, density_non_active_chr4, alpha=0.3, color='gray', label='Non-active')
ax.plot(x_range_chr4, density_non_active_chr4, color='gray', linewidth=1.5, alpha=0.8)

ax.fill_between(x_range_chr4, density_active_chr4, alpha=0.3, color='steelblue', label='Active')
ax.plot(x_range_chr4, density_active_chr4, color='steelblue', linewidth=1.5, alpha=0.8)

ax.fill_between(x_range_chr4, density_diff_chr4, alpha=0.3, color='tomato', label='Differential-active')
ax.plot(x_range_chr4, density_diff_chr4, color='tomato', linewidth=1.5, alpha=0.8)

# Highlight EVC gene region
ax.axvspan(evc_start, evc_end, alpha=0.15, color='green', label='EVC gene')

# Add rug plot for individual variants at bottom
y_lim = ax.get_ylim()
y_rug = y_lim[0] - (y_lim[1] - y_lim[0]) * 0.05

ax.scatter(non_active_positions_chr4, [y_rug]*len(non_active_positions_chr4), color='gray', s=10, alpha=0.3, marker='_', linewidths=0)
ax.scatter(active_positions_chr4, [y_rug]*len(active_positions_chr4), color='steelblue', s=20, alpha=0.5, marker='o', linewidths=0)
ax.scatter(diff_positions_chr4, [y_rug]*len(diff_positions_chr4), color='tomato', s=20, alpha=0.5, marker='o', linewidths=0)

# Update y-axis to accommodate the rug plot
ax.set_ylim(y_lim[0] - (y_lim[1] - y_lim[0]) * 0.12, y_lim[1])

# Format axes
ax.set_xlabel('Genomic Position (chr4)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Variant Density Across Chromosome 4', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Format x-axis with comma-separated thousands
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
total_active_oligos = EVC_region['a'].values+EVC_region['b'].values
print(f'total oligos in EVC window:{total_active_oligos}')
print(f'total diff active oligos in EVC window:{EVC_region['a']}')

In [ ]:
bins_sorted.to_csv("/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/genomic_scan_enrichment.tsv",sep='\t')